In [ ]:
import datetime

try:
    from zoneinfo import ZoneInfo
except ImportError:
    from backports.zoneinfo import ZoneInfo

import dotenv
import pandas

pandas.options.display.max_rows = 1000
import plotly.graph_objects as go
from scipy import stats

import dijido

dotenv.load_dotenv()
dijido.login()

# Logical day = [day 04:00 local, next day 04:00 local)
TZ = ZoneInfo("America/Los_Angeles")

PARENT_GOAL_NAME = "Update calorie target for the day ev day until 139 pounds"

# Winter span: Dec 1 through Mar 31 (adjust years if needed)
RANGE_START_DATE = datetime.date(2025, 12, 1)
RANGE_END_DATE = datetime.date(2026, 3, 31)

WITHOUT_APP_START = datetime.date(2025, 12, 1)
WITHOUT_APP_END = datetime.date(2026, 2, 28)
WITH_APP_START = datetime.date(2026, 3, 1)
WITH_APP_END = datetime.date(2026, 3, 31)


def logical_day_start(d: datetime.date) -> datetime.datetime:
    return datetime.datetime.combine(d, datetime.time(4, 0), tzinfo=TZ)


def seconds_in_logical_day(
    start: datetime.datetime,
    end: datetime.datetime,
    d: datetime.date,
) -> float:
    win_start = logical_day_start(d)
    win_end = logical_day_start(d + datetime.timedelta(days=1))
    sl = start.astimezone(TZ)
    el = end.astimezone(TZ)
    lo = max(sl, win_start)
    hi = min(el, win_end)
    if hi <= lo:
        return 0.0
    return (hi - lo).total_seconds()


def parse_activity_time(s: str) -> datetime.datetime:
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    return datetime.datetime.fromisoformat(s)


def daterange_inclusive(a: datetime.date, b: datetime.date):
    d = a
    one = datetime.timedelta(days=1)
    while d <= b:
        yield d
        d += one

In [ ]:
parent_goals = dijido.get_goals_by_name(PARENT_GOAL_NAME)
if not parent_goals:
    raise ValueError(f"No goal found named {PARENT_GOAL_NAME!r}")

if len(parent_goals) > 1:
    print(f"Warning: {len(parent_goals)} goals match the name; merging children from all.")

child_goals = []
seen = set()
for p in parent_goals:
    kids = dijido.get_goals_by_parent_id(p["_id"]) or []
    for g in kids:
        gid = g["_id"]
        if gid not in seen:
            seen.add(gid)
            child_goals.append(g)

print(f"Parent goal(s): {[p['name'] for p in parent_goals]}")
print(f"Child goals ({len(child_goals)}):")
for g in child_goals:
    print(f"  - {g['name']} ({g['_id']})")

In [ ]:
all_activities = []
for g in child_goals:
    acts = dijido.get_activities_for_goal(g["_id"], recursive=False)
    all_activities.extend(acts)

print(f"Fetched {len(all_activities)} activities across child goals")

clip_start = logical_day_start(RANGE_START_DATE)
clip_end = logical_day_start(RANGE_END_DATE + datetime.timedelta(days=1))

parsed = []
for a in all_activities:
    st = parse_activity_time(a["start_time"])
    et = parse_activity_time(a["end_time"])
    if et <= st:
        continue
    if et <= clip_start or st >= clip_end:
        continue
    parsed.append({"start": st, "end": et, "goal_id": a.get("goal_id")})

In [ ]:
def logical_date_containing_instant(t: datetime.datetime) -> datetime.date:
    return (t.astimezone(TZ) - datetime.timedelta(hours=4)).date()


daily_seconds = {d: 0.0 for d in daterange_inclusive(RANGE_START_DATE, RANGE_END_DATE)}

for row in parsed:
    st, et = row["start"], row["end"]
    et_eff = et - datetime.timedelta(microseconds=1)
    d0 = logical_date_containing_instant(st)
    d1 = logical_date_containing_instant(et_eff)
    d_start = max(d0, RANGE_START_DATE)
    d_end = min(d1, RANGE_END_DATE)
    if d_start > d_end:
        continue
    d = d_start
    while d <= d_end:
        daily_seconds[d] += seconds_in_logical_day(st, et, d)
        d += datetime.timedelta(days=1)

daily_df = pandas.DataFrame(
    [{"logical_date": d, "seconds": daily_seconds[d], "hours": daily_seconds[d] / 3600.0} for d in sorted(daily_seconds.keys())]
)

# Text table in stdout (print(df) truncates and is not aligned like a table)
print(daily_df.to_string(index=False))

In [ ]:
without_mask = (daily_df["logical_date"] >= WITHOUT_APP_START) & (daily_df["logical_date"] <= WITHOUT_APP_END)
with_mask = (daily_df["logical_date"] >= WITH_APP_START) & (daily_df["logical_date"] <= WITH_APP_END)

without_hours = daily_df.loc[without_mask, "hours"].values
with_hours = daily_df.loc[with_mask, "hours"].values

# Add a mask to remove outliers (hours > 0.5)
without_hours = without_hours[without_hours < 0.5]
with_hours = with_hours[with_hours < 0.5]

In [ ]:
# Welch's t-test (unequal variance) on daily hours; zeros included in both samples
tt = stats.ttest_ind(without_hours, with_hours, equal_var=False)
print(f"without app: n={len(without_hours)}, mean={without_hours.mean():.4f} h/day, std={without_hours.std():.4f}")
print(f"with app:    n={len(with_hours)}, mean={with_hours.mean():.4f} h/day, std={with_hours.std():.4f}")
print(f"Welch t-statistic: {tt.statistic:.4f}")
print(f"p-value (two-sided): {tt.pvalue:.6g}")

In [ ]:
# Calculate the average time saved per day, week, month, year. Calculate
# The mean of each first, then compare means

without_hours_mean = without_hours.mean()
with_hours_mean = with_hours.mean()

daily_savings = without_hours_mean - with_hours_mean
weekly_savings = daily_savings * 7
monthly_savings = daily_savings * 30
yearly_savings = daily_savings * 365

df = pandas.DataFrame(
    [
        {
            "daily_savings": daily_savings,
            "weekly_savings": weekly_savings,
            "monthly_savings": monthly_savings,
            "yearly_savings": yearly_savings,
        }
    ]
)

df

In [ ]:
delta_mean_hr = float(without_hours.mean() - with_hours.mean())
pval = float(tt.pvalue)
p_text = f"{pval:.2e}" if pval < 1e-4 else f"{pval:.4g}"
stats_text = f"Δ mean (Existing − Custom) = {delta_mean_hr:.4f} h/day  |  p = {p_text} (two-sided Welch)"

fig = go.Figure()
fig.add_trace(go.Box(y=without_hours, name="Existing App", boxpoints="all"))
fig.add_trace(go.Box(y=with_hours, name="Custom App", boxpoints="all"))

fig.add_shape(
    type="line",
    xref="paper",
    yref="paper",
    x0=0.25,
    x1=0.75,
    y0=1.035,
    y1=1.035,
    line=dict(color="rgba(42, 63, 95, 0.55)", width=1.5),
)
fig.add_annotation(
    text=stats_text,
    xref="paper",
    yref="paper",
    x=0.5,
    y=1.075,
    showarrow=False,
    xanchor="center",
    yanchor="bottom",
    font=dict(size=12, color="#2a3f5f"),
)
fig.add_annotation(
    text="Custom app reduces data entry by 19 hours per year",
    xref="paper",
    yref="paper",
    x=0.99,
    y=0.99,
    showarrow=False,
    xanchor="right",
    yanchor="top",
    align="right",
    bgcolor="rgba(255, 255, 255, 0.92)",
    bordercolor="rgba(42, 63, 95, 0.35)",
    borderwidth=1,
    borderpad=8,
    font=dict(size=11, color="#2a3f5f"),
)

fig.update_layout(
    title="Daily time spent tracking calories",
    yaxis_title="Hours per logical day",
    showlegend=True,
    plot_bgcolor="rgba(255, 255, 255, 0)",
    legend=dict(orientation="h", yanchor="top", y=-0.14, xanchor="center", x=0.5),
    margin=dict(t=100, r=24),
    width=800
)
fig.show()